# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print("\nDataset identifier (@id):", metadata.id)
print("\nDataset keywords:", getattr(metadata, 'keywords', []))

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

Below, we list all record sets available in the dataset along with their contained fields and columns. All reference is by `@id` only.

In [ ]:
# List all record sets and their @id, then their fields/columns with @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rs in record_sets:
        print(f"Record set: {rs.id}")
        if hasattr(rs, 'fields'):
            print("  Fields:")
            for f in rs.fields:
                print(f"    {f.id}")
        if hasattr(rs, 'columns'):
            print("  Columns:")
            for c in rs.columns:
                print(f"    {c.id}")
        print('')

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis.

Use the record set and field/column `@id`s from the overview above. If there are multiple record sets, they are all loaded below.

In [ ]:
# Extract data from all available record sets, using only their @id
dataframes = {}
available_record_sets = dataset.record_sets
record_set_ids = [rs.id for rs in available_record_sets]

if not record_set_ids:
    print("No record sets with data found.")
else:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        # Custom display for empty or non-tabular record sets
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
        else:
            print(f"No records available for record set {record_set_id}.")

    # Preview the first available record set with records
    for rsid, df in dataframes.items():
        print(f"\nColumns for record set {rsid}:")
        print(df.columns.tolist())
        display(df.head())
        break

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. **All columns/fields referenced by their `@id`.**

For demonstration, we select a numeric field (by its @id, e.g., `'log_likelihood'`) and a potential group field (e.g., `'ward'`). Adjust these IDs to suit actual content if necessary.

In [ ]:
# Select the first DataFrame to work with
if dataframes:
    # Pick the first record set as example
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    print(f"Working with data from record set: {record_set_id}")

    # Show all column @id for selection
    print("Available fields/columns (@id):", df.columns.tolist())

    # Example: use 'log_likelihood' and 'ward' if present
    numeric_field = None
    group_field = None
    for col in df.columns:
        if 'log_likelihood' in col:
            numeric_field = col
        if 'ward' in col.lower() or 'group' in col.lower() or 'county' in col.lower():
            group_field = col

    # If not found, pick the first numeric-looking column for demonstration
    if not numeric_field:
        for col in df.columns:
            try:
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
            except Exception:
                continue

    if numeric_field:
        # Drop NA for the numeric field
        df_clean = df.dropna(subset=[numeric_field])

        threshold = df_clean[numeric_field].mean() if pd.api.types.is_numeric_dtype(df_clean[numeric_field]) else 0
        filtered_df = df_clean[df_clean[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No tabular dataframes to explore.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset, using field/column `@id`s for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, adhering to the Croissant schema with all references by `@id`.

- **Data loaded by Croissant schema URL**, ensuring semantic access and reproducibility.
- Field and column IDs were used to reference entities in all analysis operations.
- Example EDA showed filtering and normalization by a numeric field, with grouping and basic visualization using discovered `@id`s.

For advanced analytics, users are encouraged to explore all available fields and record sets using their `@id`s as discovered in Section 2.
